# Analytics validation
Each function is checked against a closed-form or textbook value. Asserts fail the notebook if a check breaks.

In [1]:
import sys; sys.path.insert(0, '../src')
from datetime import date
import numpy as np
from muni_monitor import analytics as an
S, M = date(2020,1,15), date(2030,1,15)

## Price: 10y 6% semiannual at 6% yield = par


In [2]:
p = an.price_from_yield(S, M, 0.06, 0.06)['clean']; print(p); assert abs(p-100) < 1e-9

99.99999999999997


## Price: 10y 8% at 6% yield = 4*a(20,3%) + 100*1.03^-20 = 114.877

In [3]:
p = an.price_from_yield(S, M, 0.08, 0.06)['clean']; print(round(p,4)); assert abs(p-114.877) < 1e-3

114.8775


## Yield from price (round trip, off-coupon date)

In [4]:
st = date(2021,3,10)
px = an.price_from_yield(st, M, 0.05, 0.0375)['clean']
y = an.yield_from_price(st, M, 0.05, px); print(y); assert abs(y-0.0375) < 1e-9

0.03750000000000273


## Duration: par bond modified duration = (1 - 1.03^-20)/0.06 = 7.4387

In [5]:
r = an.risk_measures(S, M, 0.06, 0.06); print(r); assert abs(r['modified']-(1-1.03**-20)/0.06) < 1e-9

{'macaulay': 7.661899553134585, 'modified': 7.438737430227753, 'convexity': 68.77482237123748, 'dv01': np.float64(0.07438737430227751)}


## DV01 and convexity vs finite differences

In [6]:
y,h = 0.045,1e-5; f = lambda v: an.price_from_yield(S,M,0.05,v)['dirty']
r = an.risk_measures(S,M,0.05,y)
assert abs(r['dv01']-(f(y-1e-4)-f(y+1e-4))/2) < 1e-4*r['dv01']
assert abs(r['convexity']-(f(y+h)-2*f(y)+f(y-h))/(h*h*f(y))) < 1e-3*r['convexity']; print('ok', r)

ok {'macaulay': 8.035563630859837, 'modified': 7.858741937271234, 'convexity': 74.55061498032138, 'dv01': np.float64(0.08172378676962404)}


## Zero coupon: Macaulay duration equals maturity

In [7]:
assert abs(an.risk_measures(S, M, 0.0, 0.05)['macaulay']-10) < 1e-9; print('ok')

ok


## Curve interpolation: hits knots, monotone between them

In [8]:
x=np.array([1,2,5,10,30.]); y=np.array([4.0,4.2,4.5,4.8,5.2])
assert abs(an.interpolate_curve(x,y,5)-4.5)<1e-12
v=[an.interpolate_curve(x,y,t) for t in np.linspace(1,30,100)]; assert all(b>=a-1e-12 for a,b in zip(v,v[1:])); print('ok')

ok


## Spread, ratio, tax-equivalent yield

In [9]:
assert abs(an.spread_bps(0.0425,0.04)-25)<1e-9
assert abs(an.muni_treasury_ratio(0.034,0.04)-0.85)<1e-12
print(an.tax_equivalent_yield(0.04,0.37), an.tax_equivalent_yield(0.04,0.24,an.IL_INCOME_TAX))

0.06349206349206349 0.05629838142153413
